<a href="https://colab.research.google.com/github/Muskan-Kandel/aes/blob/main/AES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Set Up Folder Structure

In [ ]:
import os

# Create project folders in Colab's session storage
folders = ['/content/AES_Project/data',
           '/content/AES_Project/models',
           '/content/AES_Project/src',
           '/content/AES_Project/outputs']

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print(" Folder structure created!")

 Folder structure created!


### Install All Required Libraries
 Colab has many libraries pre-installed, so this only adds what's missing:

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes
!pip install -q rouge-score bert-score
!pip install -q nltk spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 97.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### Verify GPU is Working

In [ ]:
import torch
import transformers

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
print("Transformers version:", transformers.__version__)

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Transformers version: 5.3.0


You should see CUDA available: True and Tesla T4 as the GPU name.

In [ ]:
import os
import pandas as pd
import gdown


os.makedirs('/content/AES_Project/data', exist_ok=True)


file_ids = {
    'training_set_rel3.csv': '1dBGoTw8F2Gcgj4_b2J94EPxzqjF16gIF', # The ASAP raw data
    'Prompt-3.csv': '1H5ZiBPfboAan_EZAlDt6mTWMMrAC7b47',
    'Prompt-4.csv': '1qvw_VgpTfROzWGa2-3CgirCyN669uuBo',
    'Prompt-5.csv': '1OS0EgyEcPcUKRthtF262l_-43b0W_ZjN',
    'Prompt-6.csv': '1Guar_mZzFtBbNEIZobQvc72rEAF0PA1L'
}

print("Downloading dataset files...")
for filename, file_id in file_ids.items():
    url = f'https://drive.google.com/uc?id={file_id}'
    output = f'/content/AES_Project/data/{filename}'
    gdown.download(url, output, quiet=False)

print("All files ready in /content/AES_Project/data/")

Downloading...
From: https://drive.google.com/uc?id=1dBGoTw8F2Gcgj4_b2J94EPxzqjF16gIF
To: /content/AES_Project/data/training_set_rel3.csv
100%|██████████| 16.4M/16.4M [00:00<00:00, 94.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1H5ZiBPfboAan_EZAlDt6mTWMMrAC7b47
To: /content/AES_Project/data/Prompt-3.csv
100%|██████████| 22.5k/22.5k [00:00<00:00, 43.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1qvw_VgpTfROzWGa2-3CgirCyN669uuBo
To: /content/AES_Project/data/Prompt-4.csv
100%|██████████| 23.7k/23.7k [00:00<00:00, 54.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1OS0EgyEcPcUKRthtF262l_-43b0W_ZjN
To: /content/AES_Project/data/Prompt-5.csv
100%|██████████| 25.3k/25.3k [00:00<00:00, 42.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Guar_mZzFtBbNEIZobQvc72rEAF0PA1L
To: /content/AES_Project/data/Prompt-6.csv
100%|██████████| 25.3k/25.3k [00:00<00:00, 52.8MB/s]

All files ready in /content/AES_Project/data/


In [ ]:
asap_raw = pd.read_csv('/content/AES_Project/data/training_set_rel3.csv', encoding='latin1')

prompt3 = pd.read_csv('/content/AES_Project/data/Prompt-3.csv', encoding='latin1')

prompt4 = pd.read_csv('/content/AES_Project/data/Prompt-4.csv', encoding='latin1')

prompt5 = pd.read_csv('/content/AES_Project/data/Prompt-5.csv', encoding='latin1')

prompt6 = pd.read_csv('/content/AES_Project/data/Prompt-6.csv', encoding='latin1')


In [ ]:
!pip install -U bitsandbytes accelerate transformers
#for getting. the latest available version of bitsandbytes.
#run this only if the next block of code shows importerror, typically happens when session restarts so run each time with a new runtime

In [ ]:
#torch -> for building loss functions and training
#transformers -> for mistraal/gpt type models

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

#4-bit configuration for memory efficiency

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name) #turns text to numbers that the model understands
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bnb_config,
                                             device_map="auto")

#this block takes a little longer to load, maybe like ~ 6-7 minutes


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
filtered= asap_raw[asap_raw['essay_set'].isin([3,4,5,6])]
print("Total rows in prompt 3-6 is", len(filtered))
print("Columns : ", list(filtered.columns))
print("Per prompt : ", filtered['essay_set'].value_counts().sort_index().to_dict())
for i in [3,4,5,6]:
  p = pd.read_csv(f'/content/AES_Project/data/Prompt-{i}.csv')
  print(f'prompt-{i}.csv rows: {len(p)}, columns: {list(p.columns)}')

  #check for essay_id overlap in asap dataset and prompt 3-6 datasets

  overlap = filtered[filtered['essay_set'] == i]['essay_id'].isin(p['Essay ID']).sum()
  print(f'Matching essay_ids: {overlap}/{len(p)}')

Total rows in prompt 3-6 is 7101
Columns :  ['essay_id', 'essay_set', 'essay', 'rater1_domain1', 'rater2_domain1', 'rater3_domain1', 'domain1_score', 'rater1_domain2', 'rater2_domain2', 'domain2_score', 'rater1_trait1', 'rater1_trait2', 'rater1_trait3', 'rater1_trait4', 'rater1_trait5', 'rater1_trait6', 'rater2_trait1', 'rater2_trait2', 'rater2_trait3', 'rater2_trait4', 'rater2_trait5', 'rater2_trait6', 'rater3_trait1', 'rater3_trait2', 'rater3_trait3', 'rater3_trait4', 'rater3_trait5', 'rater3_trait6']
Per prompt :  {3: 1726, 4: 1770, 5: 1805, 6: 1800}
prompt-3.csv rows: 1726, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1726/1726
prompt-4.csv rows: 1772, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1770/1772
prompt-5.csv rows: 1805, columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Matching essay_ids: 1805/1805
prompt-6.csv rows: 1800, columns: [

In [ ]:
import pandas as pd

all_merged = []

for i in [3, 4, 5, 6]:
    #loading labels for the specific prompt
    p_labels = pd.read_csv(f'/content/AES_Project/data/Prompt-{i}.csv', encoding='latin1')

    #getting essays
    p_essays = filtered[filtered['essay_set'] == i][['essay_id', 'essay', 'domain1_score']]

    #merging on essay_id (asap uses essay_id and prompts use Essay ID)
    temp = pd.merge(p_essays, p_labels, left_on='essay_id', right_on='Essay ID')
    all_merged.append(temp)

#master dataframe
df_master = pd.concat(all_merged, ignore_index=True)

#standardizing column names
df_master = df_master.rename(columns={
    'domain1_score': 'holistic',
    'Content': 'content',
    'Prompt Adherence': 'adherence',
    'Language': 'language',
    'Narrativity': 'narrativity'
}).drop(columns=['Essay ID'])

print(f"Master Dataframe built: {len(df_master)} rows.")

Master Dataframe built: 7101 rows.


In [ ]:
import torch
import torch.nn as nn
import re
from datasets import Dataset


#format essays fro model input
def format_example(row):
  prompt = f"""

  Essay: {row['essay']}

  Score the essay with values 0-10 for:
  Content
  Language
  Adherence
  Narrativity
  Holistic

  Format:
  Content:<score>
  Language:<score>
  Adherence:<score>
  Narrativity:<score>
  Holistic:<score>

"""
  answer = f"""

  Content: {row['content']}
  Language: {row['language']}
  Adherence: {row['adherence']}
  Narrativity: {row['narrativity']}
  Holistic: {row['holistic']}

  """
  return prompt + answer

df_master["text"] = df_master.apply(format_example, axis = 1)



In [ ]:
dataset = Dataset.from_pandas(df_master[["text"]])

def tokenize_function(example):
  tokens = tokenizer(
      example["text"],
      truncation = True,
      padding = "max_length",
      max_length = 512
  )

  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/7101 [00:00<?, ? examples/s]

In [ ]:
#checking predictions and mse for first few essays:
loss_fn = nn.MSELoss()

for i, row in df_master.iterrows():
    essay = row['essay']

    messages = [{
        "role": "user",
        "content": f"""
Essay: {essay}

Carefully analyze the essay.
Then give scores (0-10) for:

Content
Language
Adherence
Narrativity
Holistic

Provide reasoning for each score.

Format strictly:

Content: <score>
Language: <score>
Adherence: <score>
Narrativity: <score>
Holistic: <score>
Reasoning: <text>
"""
    }]

    #tokenizer
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    #output generation
    outputs = model.generate(
        **inputs,
        max_new_tokens=300
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    #scores extraction
    scores = re.findall(r'(?:Content|Language|Adherence|Narrativity|Holistic)\s*:\s*(\d+)', result)

    if len(scores) == 5:
        scores = torch.tensor([int(s) for s in scores], dtype=torch.float)
        true_scores = torch.tensor([
            row['content'],
            row['language'],
            row['adherence'],
            row['narrativity'],
            row['holistic']
        ], dtype=torch.float)
        loss = loss_fn(scores, true_scores)
        print(f"Essay {i} - Loss: {loss.item()} | Pred: {scores.tolist()} | True: {true_scores.tolist()}")
        print("Model Output with Reasoning:\n", result, "\n")
    else:
        print(f"Essay {i} - Could not parse scores properly.\nFull Output:\n{result}\n")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Essay 0 - Loss: 44.0 | Pred: [7.0, 8.0, 9.0, 5.0, 6.0] | True: [0.0, 1.0, 0.0, 1.0, 1.0]
Model Output with Reasoning:
 [INST] 
Essay: The features of the setting affect the cyclist in many ways. The features of the setting that affect the cyclist is the lact of information on were to go and the lack of water. This was a problem because he needed water for his trip and directions on were to go.  

Carefully analyze the essay.
Then give scores (0-10) for:

Content
Language
Adherence
Narrativity
Holistic

Provide reasoning for each score.

Format strictly:

Content: <score>
Language: <score>
Adherence: <score>
Narrativity: <score>
Holistic: <score>
Reasoning: <text>
 [/INST] Content: 7
The essay provides a clear issue that affects the cyclist, which are the lack of information and water. However, it could benefit from more specific details about the setting and the cyclist's journey.

Language: 8
The language used in the essay is clear and concise, making it easy to understand the main po

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Essay 1 - Loss: 46.400001525878906 | Pred: [10.0, 9.0, 10.0, 8.0, 9.0] | True: [3.0, 2.0, 3.0, 2.0, 2.0]
Model Output with Reasoning:
 [INST] 
Essay: The features of the setting affected the cyclist in a negative way. He was in the desert, which is dry and hot. âI was traveling through the high deserts of California in June."(Kurmaskie @NUM1). Also, the  temperature was hot and made every thing around him hot, âbrackish water faling somewhere in the neighborhood of two hundred degrees" (Kurmaskie @NUM1). Since he was in the desert, there was no escape from the sun. The text  states, âThe sun was beginning to beat downâ(Kurmaskie @NUM3). The cyclist was all alone in the road with none around him either so into he passed out, no one could help him. â and the growing realization that I could drop from heatstroke on a gorgous day in Juneâ(Kurmaskie @NUM1). "About forty miles into the pedal, I arrived at the first âtownâ but on that morning it fit the traditional definition 

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Essay 5 - Loss: 42.0 | Pred: [8.0, 7.0, 9.0, 6.0, 7.0] | True: [1.0, 1.0, 1.0, 1.0, 1.0]
Model Output with Reasoning:
 [INST] 
Essay: There were many features of the setting that affected the cyclist. For instance, it was extremely hot and dry. Also it  was nearly deserted, almost all of the places he crossed were deserted .A  lost her features of the setting were a ram shackle shed, several rusty pumps, and a corral that caldnt hald in the lamest move.an is the feature give you an idea if haw the cyclist feels.  

Carefully analyze the essay.
Then give scores (0-10) for:

Content
Language
Adherence
Narrativity
Holistic

Provide reasoning for each score.

Format strictly:

Content: <score>
Language: <score>
Adherence: <score>
Narrativity: <score>
Holistic: <score>
Reasoning: <text>
 [/INST] Content: 8
The essay provides a clear description of the setting and its impact on the cyclist. It mentions several features of the setting, such as the heat, dryness, desertedness, and the specific

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Essay 6 - Loss: 8.399999618530273 | Pred: [3.0, 5.0, 2.0, 4.0, 3.0] | True: [0.0, 1.0, 0.0, 1.0, 1.0]
Model Output with Reasoning:
 [INST] 
Essay: The cyclist was riding through a tower when he stopped for directions. These old men told him to take a short cut and he did and got lost. He was in a ghost town in the desert. In conclusion the desert is hot.

Carefully analyze the essay.
Then give scores (0-10) for:

Content
Language
Adherence
Narrativity
Holistic

Provide reasoning for each score.

Format strictly:

Content: <score>
Language: <score>
Adherence: <score>
Narrativity: <score>
Holistic: <score>
Reasoning: <text>
 [/INST] Content: 3
The essay provides very little information beyond the fact that a cyclist got lost in a desert ghost town. There is no clear beginning, middle, or end to the story, and no context is given about why the cyclist was in the area or how he ended up in the ghost town. The statement about the desert being hot is also not directly related to the content 

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Essay 7 - Loss: 18.200000762939453 | Pred: [5.0, 4.0, 3.0, 5.0, 4.0] | True: [0.0, 0.0, 0.0, 0.0, 0.0]
Model Output with Reasoning:
 [INST] 
Essay: The affects of the cyclist is if it does not change. it cut hurt a lot of people feeling because they @CAPS1 donât care about the cyclist. Iâm one of them people who does not care about it cause it does not affect me or anyone I know. It is a big deal to write people, some of them @CAPS1 blow stuff up. They talking on tiv and on the radio making all this stuff they say is made up. I donât believe to I see it. That @CAPS1 not for black people because it has not did anything to us. We really donât care about the affects of the cyclist.

Carefully analyze the essay.
Then give scores (0-10) for:

Content
Language
Adherence
Narrativity
Holistic

Provide reasoning for each score.

Format strictly:

Content: <score>
Language: <score>
Adherence: <score>
Narrativity: <score>
Holistic: <score>
Reasoning: <text>
 [/INST] Content: 5
The essay e